# Three kinds of diagnosis

After you measure the leftover of a vector, look one degree up. Two numbers decide the label:

- $\Phi_1$: cheapest leftover after licensed first-stage repairs.
- $\Gamma_2$: cheapest leftover of the *next* defect after licensed second-stage repairs.

Every well-posed input gets exactly one of three labels.

| Label | Meaning in words |
|---|---|
| `ExactRepair` | You can cancel the vector completely. $\Phi_1=0$. |
| `CoherentResidual` | A leftover remains, but the next stage can be cancelled. $\Phi_1>0$ and $\Gamma_2=0$. |
| `NonclosedDefect` | Leftover remains, and it still shows one degree up. $\Phi_1>0$ and $\Gamma_2>0$. |

The labels classify the vector. They do not, by themselves, choose an action.

Run from the package root, or from this `jupyter_notebooks/` folder.

In [1]:
from pathlib import Path
import sys

p = Path.cwd().resolve()
if p.name == "jupyter_notebooks":
    root = p.parent
elif (p / "code" / "finite_obstruction.py").is_file():
    root = p
elif (p / "coda_atlas" / "F005_coda" / "foc" / "code").is_dir():
    root = p / "coda_atlas" / "F005_coda" / "foc"
elif (p / "foc" / "code").is_dir():
    root = p / "foc"
else:
    root = p
assert (root / "code").is_dir(), "run from the package root or jupyter_notebooks/"
if str(root / "code") not in sys.path:
    sys.path.insert(0, str(root / "code"))

from sympy import Matrix, Rational
from finite_obstruction import FiniteObstructionPacket, diagnose

## ExactRepair

Same plane as before: repairs along $(1,0)$, next-defect map zero. The vector $(3,0)$ is on the repair line, so the leftover is $0$.

In [2]:
closed = FiniteObstructionPacket.create(
    d0=[[1], [0]],
    d1=[[0, 0]],
    a2=Matrix.zeros(1, 0),
    w1=[1, 1],
    w2=[1],
)
exact = diagnose(closed, [3, 0])
print(exact.diagnostic.value, "Phi1 =", exact.phi1.value, "Gamma2 =", exact.gamma2.value)
assert exact.diagnostic.value == "ExactRepair"

ExactRepair Phi1 = 0 Gamma2 = 0


## CoherentResidual

Keep the same packet. The vector $(0,4)$ cannot be cancelled, so $\Phi_1=4$. The next-defect map is still zero, so $\Gamma_2=0$. A leftover remains, but it does not leak.

In [3]:
coherent = diagnose(closed, [0, 4])
print(coherent.diagnostic.value, "Phi1 =", coherent.phi1.value, "Gamma2 =", coherent.gamma2.value)
assert coherent.diagnostic.value == "CoherentResidual"
assert coherent.phi1.value == Rational(4)
assert coherent.gamma2.value == Rational(0)

CoherentResidual Phi1 = 4 Gamma2 = 0


## NonclosedDefect

Change only the next-defect map: let it read the second coordinate. Repairs still cannot touch that coordinate, so the leftover $4$ is copied one degree up. There are no second-stage repairs, so $\Gamma_2$ is also positive.

In [4]:
leaky = FiniteObstructionPacket.create(
    d0=[[1], [0]],
    d1=[[0, 1]],
    a2=Matrix.zeros(1, 0),
    w1=[1, 1],
    w2=[2],
)
nonclosed = diagnose(leaky, [0, 4])
print(nonclosed.diagnostic.value, "Phi1 =", nonclosed.phi1.value, "Gamma2 =", nonclosed.gamma2.value)
assert nonclosed.diagnostic.value == "NonclosedDefect"
assert nonclosed.phi1.value == Rational(4)
assert nonclosed.gamma2.value > 0

NonclosedDefect Phi1 = 4 Gamma2 = 8
